In [2]:
# Tokenization

# Tokenization for previous maodel - chars (all the possible character in mini shakesphere dataset)

# What are token to a LLM?
# Token: a small chunk of text (a character, part of a word, or a whole word)
# that the model treats as a single unit, represented by an integer ID.

In [3]:
# One of the popular mechanism for tokenization is Byte pair encoding
# from paper - https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf
# read the input representation part of the paper

In [4]:
# Tricky -> cause there are many languages and other symbols like special chars and emogi's

In [9]:
[ord(x) for x in "안녕하세요 👋 (hello in Korean!)"]

[50504,
 45397,
 54616,
 49464,
 50836,
 32,
 128075,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

In [11]:
list("안녕하세요 👋 (hello in Korean!)".encode('utf8')) # utf-16 and utf-32 are very wasteful 

[236,
 149,
 136,
 235,
 133,
 149,
 237,
 149,
 152,
 236,
 132,
 184,
 236,
 154,
 148,
 32,
 240,
 159,
 145,
 139,
 32,
 40,
 104,
 101,
 108,
 108,
 111,
 32,
 105,
 110,
 32,
 75,
 111,
 114,
 101,
 97,
 110,
 33,
 41]

In [12]:
# Why not just use raw utf-8?

# then we only have 0-265 vocab, so small encoding vector, but the same string will be represented in a much larger
# sequence then using chunks. which is not computationally or cost effective

In [82]:
# how byte pair encoding works?
# In a given sequence - we iteratively find out the pairs that occurs frequently 
# and represent then using a small symbolic repr (that decresase the sequence size but increases our 
# vocab size) till we are satisfied

# see - https://en.wikipedia.org/wiki/Byte-pair_encoding

In [136]:
# text from https://www.reedbeta.com/blog/programmers-intro-to-unicode/
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = text.encode('utf8') # raw bytes
tokens = list(map(int, tokens)) # convert to a list of ints in range 0...255

len(tokens)

616

In [137]:
# what we want to do, iterate over tokens and find the pair that occures most frequently, then we merge it
from collections import defaultdict

vocab_size = 276
num_merges = vocab_size - 256

def encode(tokens, num_merges):
    mint = {}
    tokens = tokens
    nextt = 256
    for _ in range(num_merges):
        
        # find the count of pairs
        pair_count = defaultdict(int)
        for i in range(1, len(tokens)):
            pair_count[(tokens[i-1], tokens[i])] += 1

        # breaks if not element or find pair wiht max coint
        if not pair_count:
            break
        to_mint = tuple(max(pair_count, key=pair_count.get))

        # replace the pair with mint
        temp = []
        j = 0
        while j < len(tokens) :
            x, y = to_mint
            if j < len(tokens) - 1 and tokens[j] == x and tokens[j+1] == y:
                temp.append(nextt)
                j += 2
            else:
                temp.append(tokens[j])
                j += 1

        # update the variables
        mint[to_mint] = nextt
        nextt += 1
        tokens = temp

    return mint, tokens

mint, encoded = encode(tokens, num_merges)

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in mint.items():
    vocab[idx] = vocab[p0] + vocab[p1]

# def decode(tokens, mint):
#     out = tokens
#     mint = reversed(mint)
#     for m, pair in mint:
#         temp = []
#         for i in out:
#             # print(i)
#             if i == m:
#                 temp.extend(list(pair))
#             else:
#                 temp.append(i)
#         out = temp
#     return out

def decode(ids):
    #given ids (list of integers) return Python string
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf8", errors='repalce')
    return text

decoded = decode(encoded)
print(decoded == text)

True


In [40]:
tokens

[300,
 32,
 379,
 118,
 105,
 357,
 357,
 288,
 364,
 97,
 103,
 101,
 352,
 112,
 384,
 356,
 317,
 101,
 369,
 317,
 115,
 117,
 112,
 112,
 384,
 287,
 116,
 97,
 403,
 101,
 120,
 101,
 402,
 101,
 385,
 308,
 115,
 323,
 32,
 110,
 111,
 116,
 371,
 275,
 98,
 101,
 352,
 116,
 116,
 108,
 376,
 333,
 32,
 73,
 32,
 379,
 110,
 226,
 338,
 108,
 97,
 315,
 97,
 315,
 287,
 115,
 317,
 291,
 116,
 333,
 108,
 380,
 105,
 357,
 288,
 364,
 116,
 371,
 111,
 397,
 118,
 101,
 369,
 51,
 48,
 32,
 121,
 101,
 97,
 403,
 401,
 110,
 336,
 293,
 270,
 101,
 385,
 333,
 110,
 46]